In [35]:
# Необходимо для корректной работы внешних .py файлов
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from helper import *
import random

In [37]:
# фиксируем состояние генератора псевдослучайных чисел. Это необходимо, чтобы результаты модели не менялись с каждым новым запуском 
SEED = 42 # можно указать любое число
np.random.seed(SEED)
random.seed(SEED)

# Загружаем датасет в память
df = pd.read_csv("diamonds.csv", index_col=0)

## 5. Препроцессинг данных

### 5.1 Общие преобразования

In [38]:
df_preprocessed = df.copy()

In [39]:
# Дубликаты
df_preprocessed.drop_duplicates(inplace=True)

In [40]:
# Удаление строк, где x/y/z = 0
df_preprocessed = df_preprocessed[(df_preprocessed[["x", "y", "z"]] != 0).all(axis=1)]

In [41]:
# Убеждаемся, что типы признаков сохранились как нам нужно
df_preprocessed.info()

<class 'pandas.DataFrame'>
Index: 53775 entries, 1 to 53940
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53775 non-null  float64
 1   cut      53775 non-null  str    
 2   color    53775 non-null  str    
 3   clarity  53775 non-null  str    
 4   depth    53775 non-null  float64
 5   table    53775 non-null  float64
 6   price    53775 non-null  int64  
 7   x        53775 non-null  float64
 8   y        53775 non-null  float64
 9   z        53775 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.5 MB


### 5.2 Разбиение датасета 

In [42]:
# делим данные на признаки и таргет
X, y = divide_data(df_preprocessed, "price")

In [43]:
# Разбиваем данные на train, val и test с соотношением 60-20-20
X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(
    X, y, 
    train_size=0.6, 
    val_size=0.2, 
    test_size=0.2, 
    random_state=SEED,
)

In [44]:
X_train_preprocessed = X_train.copy()

### 5.3 Итоговый пайплайн для препроцессинга
Для удобства объединим все наши преобразования в единый пайплайн, который можно будет впоследствии переиспользовать на тестовой выборке. Это поможет избежать излишнего копипаста, ошибок и предотваратить data leakage

In [ ]:
categorical_cols = ["cut", "color", "clarity"]
category_orders = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"], # cut
    ["D", "E", "F", "G", "H", "I", "J"], # color
    ["IF", "VVS1", "VVS2", "VS1", "VS2", "SI1", "SI2", "I1"], # clarity
]

preprocessor = Pipeline([    
    ("transformations", ColumnTransformer(
        [("encoder", OrdinalEncoder(categories=category_orders), categorical_cols)],
        remainder="passthrough",  # числовые колонки передаются без изменений
        verbose_feature_names_out=False
    ))
])

# Устанавливаем вывод в формате pandas DataFrame (для sklearn версии 1.0 и выше)
preprocessor.set_output(transform="pandas")

,steps,"[('transformations', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_

In [46]:
# обучаем и применяем наш пайплайн к тестовой выборке
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_train_preprocessed = pd.DataFrame(X_train_preprocessed, index=X_train.index)

In [47]:
# смотрим на результат работы пайплайна
X_train_preprocessed 

,cut,color,clarity,carat,depth,table,x,y,z
15430,2.0,4.0,3.0,1.20,62.0,57.0,6.77,6.81,4.21
52670,4.0,3.0,4.0,0.80,60.1,57.0,6.05,6.02,3.63
43144,2.0,4.0,4.0,0.52,63.5,58.0,5.08,5.09,3.23
32503,2.0,2.0,5.0,0.40,63.3,57.0,4.68,4.73,2.98
33373,2.0,3.0,4.0,0.41,61.7,61.0,4.70,4.74,2.91
...,...,...,...,...,...,...,...,...,...
11882,3.0,4.0,3.0,1.08,62.5,60.0,6.55,6.51,4.08
44832,3.0,1.0,5.0,0.51,61.7,59.0,5.12,5.09,3.15
3687,2.0,5.0,3.0,0.90,62.6,58.0,6.14,6.17,3.85
676,4.0,0.0,2.0,0.54,61.5,55.0,5.25,5.29,3.24


In [48]:
# снова проверим типы данных. Мы видим, что все столбцы теперь числовые, что нам и требовалось
X_train_preprocessed.info()

<class 'pandas.DataFrame'>
Index: 32265 entries, 15430 to 40652
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   cut      32265 non-null  float64
 1   color    32265 non-null  float64
 2   clarity  32265 non-null  float64
 3   carat    32265 non-null  float64
 4   depth    32265 non-null  float64
 5   table    32265 non-null  float64
 6   x        32265 non-null  float64
 7   y        32265 non-null  float64
 8   z        32265 non-null  float64
dtypes: float64(9)
memory usage: 2.5 MB
